In [2]:
from relbench.datasets import get_dataset
from relbench.tasks import get_task

dataset = get_dataset("rel-stack", download=True)
db = dataset.get_db()


100%|████████████████████████████████████████| 881M/881M [00:00<00:00, 646GB/s]
Unzipping contents of '/home/abed/.cache/relbench/rel-stack/db.zip' to '/home/abed/.cache/relbench/rel-stack/.'


Loading Database object from /home/abed/.cache/relbench/rel-stack/db...
Done in 7.48 seconds.


In [3]:
task = get_task("rel-stack", "user-engagement", download=True)
train_table = task.get_table("train")
val_table   = task.get_table("val")
test_table  = task.get_table("test")

100%|█████████████████████████████████████| 5.02M/5.02M [00:00<00:00, 4.91GB/s]
Unzipping contents of '/home/abed/.cache/relbench/rel-stack/tasks/user-engagement.zip' to '/home/abed/.cache/relbench/rel-stack/tasks/.'


In [5]:
print("Tables in DB:", list(db.table_dict.keys()))
for name, tbl in db.table_dict.items():
    print(f"\n=== {name} ===  shape={tbl.df.shape}")
    print(tbl.df.dtypes)
    print(tbl.df.head(2))


Tables in DB: ['users', 'postHistory', 'postLinks', 'votes', 'posts', 'badges', 'comments']

=== users ===  shape=(255360, 8)
Id                          Int64
AccountId                 float64
DisplayName                object
Location                   object
ProfileImageUrl           float64
WebsiteUrl                 object
AboutMe                    object
CreationDate       datetime64[ns]
dtype: object
   Id  AccountId   DisplayName            Location  ProfileImageUrl  \
0   0       -1.0     Community  on the server farm              NaN   
1   1        2.0  Geoff Dalgas       Corvallis, OR              NaN   

                       WebsiteUrl  \
0  http://meta.stackexchange.com/   
1        http://stackoverflow.com   

                                             AboutMe            CreationDate  
0  <p>Hi, I'm not really a person.</p>\n\n<p>I'm ... 2010-07-19 06:55:26.860  
1  <p>Dev #2 who helped create Stack Overflow cur... 2010-07-19 14:01:36.697  

=== postHistory ===  sha

In [7]:
for name, t in [("train", train_table), ("val", val_table), ("test", test_table)]:
    df = t.df
    print(f"{name}: shape={df.shape}, cols={list(df.columns)}")
    print(df.head(2))
    if "contribution" in df.columns:
        print("  positive rate:", df["contribution"].mean())


train: shape=(1360850, 3), cols=['timestamp', 'OwnerUserId', 'contribution']
   timestamp  OwnerUserId  contribution
0 2012-01-12          352             1
1 2010-10-14            7             1
  positive rate: 0.04998346621596796
val: shape=(85838, 3), cols=['timestamp', 'OwnerUserId', 'contribution']
   timestamp  OwnerUserId  contribution
0 2020-10-01       246773             1
1 2020-10-01       199619             1
  positive rate: 0.02808779328502528
test: shape=(88137, 2), cols=['timestamp', 'OwnerUserId']
   timestamp  OwnerUserId
0 2021-01-01          668
1 2021-01-01       125849


In [9]:
for name, tbl in db.table_dict.items():
    print(f"{name}: pkey={tbl.pkey_col}, fkeys={tbl.fkey_col_to_pkey_table}")


users: pkey=Id, fkeys={}
postHistory: pkey=Id, fkeys={'PostId': 'posts', 'UserId': 'users'}
postLinks: pkey=Id, fkeys={'PostId': 'posts', 'RelatedPostId': 'posts'}
votes: pkey=Id, fkeys={'PostId': 'posts', 'UserId': 'users'}
posts: pkey=Id, fkeys={'OwnerUserId': 'users', 'ParentId': 'posts'}
badges: pkey=Id, fkeys={'UserId': 'users'}
comments: pkey=Id, fkeys={'UserId': 'users', 'PostId': 'posts'}


## Q1 — Three manual join-aggregate features

We picked three features, each pulling from a different table in the database, to try to capture different sides of "is this user active?".

**Feature 1: `num_posts_before_t`** — number of posts the user has made before the target timestamp. Counts rows in the `posts` table where `OwnerUserId` matches the user and `CreationDate <= timestamp`. We expect users who post a lot to keep posting in the next 3 months, so this should be a positive predictor of the label.

**Feature 2: `num_votes_before_t`** — number of votes the user cast before the timestamp. Counts rows in the `votes` table where `UserId` matches and `CreationDate <= timestamp`. Voting is the easiest form of participation on Stack Overflow, so even less active users might vote, which should make this a broad activity proxy. Heads up though: it turned out a lot of vote rows have `UserId = <NA>` in the dataset because votes are anonymized, so the feature ends up very sparse - most users get 0.

**Feature 3: `days_since_last_comment`** — how many days have passed since the user's last comment before the timestamp. NaN if they never commented. Recency feels like it should matter a lot for this task: someone who commented yesterday is much more likely to engage soon than someone whose last comment was 4 years ago. The NaN case ("never commented at all") is its own meaningful bucket.

One thing we were careful about: every feature filters to `source_creation_date <= timestamp`. Without this filter, we'd be using data from after the prediction time, which leaks the label.

After running it on the train target (1,360,850 rows), here's what we saw:
- `num_posts_before_t`: mean around 3.86, max 4637 (very long-tailed distribution)
- `num_votes_before_t`: mean 0.057 with 75th percentile = 0 (sparse, because of the NaN UserId issue mentioned above)
- `days_since_last_comment`: NaN for 545,289 rows (about 40%). For users who had commented, median was around 663 days.

In [12]:
import pandas as pd
import numpy as np

users_df    = db.table_dict["users"].df
posts_df    = db.table_dict["posts"].df
votes_df    = db.table_dict["votes"].df
comments_df = db.table_dict["comments"].df

UID = "OwnerUserId"  # user-id column in the target table

def _count_before(target_df, src_df, user_col, name):
    """Count rows of src_df per (user, timestamp) where src CreationDate <= timestamp."""
    s = src_df[[user_col, "CreationDate"]].dropna(subset=[user_col]).copy()
    s[user_col] = s[user_col].astype("int64")
    s = s.rename(columns={user_col: UID})
    m = target_df[[UID, "timestamp"]].merge(s, on=UID, how="left")
    valid = m["CreationDate"].notna() & (m["CreationDate"] <= m["timestamp"])
    return (m.assign(valid=valid)
              .groupby([UID, "timestamp"])["valid"].sum()
              .astype("int64").rename(name))

def build_q1_features(target_df):
    cols = [UID, "timestamp"] + (["contribution"] if "contribution" in target_df.columns else [])
    out = target_df[cols].copy()

    # Feature 1: # posts owned by user before timestamp
    f1 = _count_before(out, posts_df, "OwnerUserId", "num_posts_before_t")

    # Feature 2: # votes cast by user before timestamp
    f2 = _count_before(out, votes_df, "UserId", "num_votes_before_t")

    # Feature 3: days since user's last comment (NaN if user never commented before t)
    c = comments_df[["UserId", "CreationDate"]].dropna(subset=["UserId"]).copy()
    c["UserId"] = c["UserId"].astype("int64")
    c = c.rename(columns={"UserId": UID})
    m = out[[UID, "timestamp"]].merge(c, on=UID, how="left")
    m = m[m["CreationDate"].notna() & (m["CreationDate"] <= m["timestamp"])]
    last_c = m.groupby([UID, "timestamp"])["CreationDate"].max().rename("last_comment_t")
    joined = out.set_index([UID, "timestamp"]).join(last_c)
    delta = joined.index.get_level_values("timestamp") - joined["last_comment_t"]
    f3 = (delta.dt.total_seconds() / 86400.0).rename("days_since_last_comment")

    out = out.set_index([UID, "timestamp"]).join([f1, f2, f3]).reset_index()
    return out

train_q1 = build_q1_features(train_table.df)
print("shape:", train_q1.shape)
print(train_q1.head())
print("\nNaN counts:\n", train_q1.isna().sum())
print("\nDescribe:\n", train_q1[["num_posts_before_t", "num_votes_before_t", "days_since_last_comment"]].describe())

shape: (1360850, 6)
   OwnerUserId  timestamp  contribution  num_posts_before_t  \
0          352 2012-01-12             1                 218   
1            7 2010-10-14             1                  80   
2         1693 2011-01-13             1                   6   
3         1160 2011-04-14             1                   7   
4         1895 2011-04-14             1                  31   

   num_votes_before_t  days_since_last_comment  
0                   2                 2.231586  
1                   0                 0.621124  
2                   0                 6.150768  
3                   0               157.357496  
4                   0                 0.272961  

NaN counts:
 OwnerUserId                     0
timestamp                       0
contribution                    0
num_posts_before_t              0
num_votes_before_t              0
days_since_last_comment    545289
dtype: int64

Describe:
        num_posts_before_t  num_votes_before_t  days_since_last_c

## Q2 — Automatic join-aggregate feature generation

Here's the algorithm we wrote (BFS/DFS over the schema FK graph, starting from `users`):

1. Build a graph from the schema. Every foreign key gives an edge, and we add the edge in both directions (so we can walk users → posts or posts → users).
2. Starting from `users`, find every simple path of length 1 up to `max_depth` edges. We don't revisit a table on the same path so we don't create cycles.
3. For each path, join all the tables along the path. The target → users join is a LEFT JOIN (so all target rows are kept). The other joins are natural joins via the FK relationship.
4. **Temporal cutoff**: when aggregating per `(OwnerUserId, timestamp)` target row, only keep joined rows whose creation_date in the leaf table is `<= timestamp`. Same idea as Q1 - prevents leaking future data.
5. For each column in the leaf table, generate aggregations based on the column type:
   - Numeric: mean, count, sum
   - Text or categorical: count
   - Timestamp: min, max
6. Concatenate all the features across all paths into one big table.

We also skip the PK column and FK columns when aggregating, since those are just IDs and don't carry meaningful values to aggregate.

In [13]:
from collections import defaultdict
import pandas as pd
import numpy as np

# ---------- schema graph ----------
def build_schema_graph(db):
    """Each edge: (neighbor_table, col_in_curr, col_in_neighbor, edge_label)."""
    schema = db.table_dict
    adj = defaultdict(list)
    for tname, tbl in schema.items():
        for fk_col, ref_table in tbl.fkey_col_to_pkey_table.items():
            ref_pk = schema[ref_table].pkey_col
            # forward edge: tname --(fk_col -> ref_pk)--> ref_table
            adj[tname].append((ref_table, fk_col, ref_pk, f"{tname}.{fk_col}->{ref_table}.{ref_pk}"))
            # reverse edge: ref_table --(ref_pk -> fk_col)--> tname
            adj[ref_table].append((tname, ref_pk, fk_col, f"{ref_table}.{ref_pk}->{tname}.{fk_col}"))
    return adj

def enumerate_paths(adj, start, max_depth):
    """All simple paths starting at `start` of length 1..max_depth.
    Each path is a list of edges (neighbor, col_in_prev, col_in_neighbor, label)."""
    out = []
    def dfs(curr, visited, edges):
        if len(edges) >= 1:
            out.append(list(edges))
        if len(edges) >= max_depth:
            return
        for (nxt, lcol, rcol, lbl) in adj[curr]:
            if nxt in visited: 
                continue
            visited.add(nxt)
            edges.append((nxt, lcol, rcol, lbl))
            dfs(nxt, visited, edges)
            edges.pop()
            visited.remove(nxt)
    dfs(start, {start}, [])
    return out

# ---------- per-table info: which column is the "creation time", and which cols are aggregable ----------
TIME_COL = {
    "users": "CreationDate", "posts": "CreationDate", "postHistory": "CreationDate",
    "postLinks": "CreationDate", "votes": "CreationDate", "comments": "CreationDate",
    "badges": "Date",
}

def aggregable_columns(db, table_name):
    """Return list of (col_name, kind) for columns we will aggregate. Skip PK & FK columns."""
    tbl = db.table_dict[table_name]
    skip = {tbl.pkey_col} | set(tbl.fkey_col_to_pkey_table.keys())
    out = []
    for c, dtype in tbl.df.dtypes.items():
        if c in skip: 
            continue
        if pd.api.types.is_datetime64_any_dtype(dtype):
            out.append((c, "time"))
        elif pd.api.types.is_numeric_dtype(dtype) or pd.api.types.is_bool_dtype(dtype):
            out.append((c, "num"))
        else:
            out.append((c, "text"))
    return out

def count_features_for_depth(db, max_depth):
    adj = build_schema_graph(db)
    paths = enumerate_paths(adj, "users", max_depth - 1)  # depth-1 here = target⟕users (0 edges from users)
    # depth-1 (the assignment's "1 hop") corresponds to ZERO edges from `users` => just the users table itself
    n = 0
    # depth-1: the "users" path
    cols = aggregable_columns(db, "users")
    n += sum(3 if k == "num" else (1 if k == "text" else 2) for _, k in cols)
    # depth-d for d >= 2: paths of length d-1 from users
    for edges in paths:
        leaf = edges[-1][0]
        cols = aggregable_columns(db, leaf)
        n += sum(3 if k == "num" else (1 if k == "text" else 2) for _, k in cols)
    return n

# sanity check
adj = build_schema_graph(db)
print("schema adjacency:")
for k, vs in adj.items():
    print(" ", k, "->", [(v[0], v[3]) for v in vs])

print("\nQ3: feature counts by depth")
for d in [1, 2, 3]:
    print(f"  max_depth={d}: {count_features_for_depth(db, d)} features")

schema adjacency:
  postHistory -> [('posts', 'postHistory.PostId->posts.Id'), ('users', 'postHistory.UserId->users.Id')]
  posts -> [('postHistory', 'posts.Id->postHistory.PostId'), ('postLinks', 'posts.Id->postLinks.PostId'), ('postLinks', 'posts.Id->postLinks.RelatedPostId'), ('votes', 'posts.Id->votes.PostId'), ('users', 'posts.OwnerUserId->users.Id'), ('posts', 'posts.ParentId->posts.Id'), ('posts', 'posts.Id->posts.ParentId'), ('comments', 'posts.Id->comments.PostId')]
  users -> [('postHistory', 'users.Id->postHistory.UserId'), ('votes', 'users.Id->votes.UserId'), ('posts', 'users.Id->posts.OwnerUserId'), ('badges', 'users.Id->badges.UserId'), ('comments', 'users.Id->comments.UserId')]
  postLinks -> [('posts', 'postLinks.PostId->posts.Id'), ('posts', 'postLinks.RelatedPostId->posts.Id')]
  votes -> [('posts', 'votes.PostId->posts.Id'), ('users', 'votes.UserId->users.Id')]
  badges -> [('users', 'badges.UserId->users.Id')]
  comments -> [('users', 'comments.UserId->users.Id'), (

In [15]:
# ----- Q2: materialize the feature_matrix at max_depth=2 -----
import time

UID = "OwnerUserId"

def _agg_dict(cols_info):
    d = {}
    for c, kind in cols_info:
        if kind == "num":
            d[c] = ["mean", "count", "sum"]
        elif kind == "text":
            d[c] = ["count"]
        else:  # time
            d[c] = ["min", "max"]
    return d

def build_feature_matrix(target_df, db, max_depth=2, verbose=True):
    assert max_depth == 2, "this implementation is specialized to depth=2"
    t0 = time.time()
    base_cols = [UID, "timestamp"] + (["contribution"] if "contribution" in target_df.columns else [])
    fm = target_df[base_cols].reset_index(drop=True).copy()
    fm["_tid"] = np.arange(len(fm), dtype=np.int64)

    # ---------- depth-1: target ⟕ users ----------
    if verbose: print("[depth-1] target ⟕ users ...")
    users_df = db.table_dict["users"].df.rename(columns={"Id": UID}).copy()
    users_df[UID] = users_df[UID].astype("int64")
    j = fm[["_tid", UID]].merge(users_df, on=UID, how="left").set_index("_tid")
    new_cols = {}
    for col, kind in aggregable_columns(db, "users"):
        if kind == "num":
            new_cols[f"users__{col}__mean"]  = j[col].astype("float64")
            new_cols[f"users__{col}__count"] = j[col].notna().astype("int64")
            new_cols[f"users__{col}__sum"]   = j[col].astype("float64")
        elif kind == "text":
            new_cols[f"users__{col}__count"] = j[col].notna().astype("int64")
        else:  # time
            new_cols[f"users__{col}__min"]   = j[col]
            new_cols[f"users__{col}__max"]   = j[col]
    d1 = pd.DataFrame(new_cols)

    # ---------- depth-2: target ⟕ users ⋈ X (for each user-neighbor X) ----------
    depth2 = [
        ("posts",       "OwnerUserId"),
        ("postHistory", "UserId"),
        ("votes",       "UserId"),
        ("badges",      "UserId"),
        ("comments",    "UserId"),
    ]
    depth2_frames = []
    for tname, user_fk in depth2:
        if verbose: print(f"[depth-2] users ⋈ {tname} (on {user_fk}) ...", end=" ")
        time_col = TIME_COL[tname]
        cols_info = aggregable_columns(db, tname)
        keep = list(dict.fromkeys([UID, time_col] + [c for c, _ in cols_info]))
        src = db.table_dict[tname].df.rename(columns={user_fk: UID})[keep].copy()
        src = src.dropna(subset=[UID])
        src[UID] = src[UID].astype("int64")

        m = fm[["_tid", UID, "timestamp"]].merge(src, on=UID, how="inner")
        m = m[m[time_col] <= m["timestamp"]]
        if verbose: print(f"matched rows: {len(m):,}")
        if len(m) == 0:
            continue

        g = m.groupby("_tid").agg(_agg_dict(cols_info))
        g.columns = [f"{tname}__{c}__{a}" for c, a in g.columns]
        depth2_frames.append(g)

    # ---------- assemble ----------
    feature_matrix = (
        fm.set_index("_tid")
          .join(d1)
          .join(depth2_frames, how="left")
    )
    count_cols = [c for c in feature_matrix.columns if c.endswith("__count")]
    feature_matrix[count_cols] = feature_matrix[count_cols].fillna(0).astype("int64")
    feature_matrix = feature_matrix.reset_index(drop=True)
    if verbose: print(f"feature_matrix shape: {feature_matrix.shape}   ({time.time()-t0:.1f}s)")
    return feature_matrix

fm_train = build_feature_matrix(train_table.df, db, max_depth=2)
print("\ncolumns ({}):".format(len(fm_train.columns)))
print(list(fm_train.columns))
fm_train.head(2)

[depth-1] target ⟕ users ...
[depth-2] users ⋈ posts (on OwnerUserId) ... matched rows: 5,250,955
[depth-2] users ⋈ postHistory (on UserId) ... matched rows: 16,814,778
[depth-2] users ⋈ votes (on UserId) ... matched rows: 77,552
[depth-2] users ⋈ badges (on UserId) ... matched rows: 5,425,886
[depth-2] users ⋈ comments (on UserId) ... matched rows: 9,705,158
feature_matrix shape: (1360850, 54)   (26.4s)

columns (54):
['OwnerUserId', 'timestamp', 'contribution', 'users__AccountId__mean', 'users__AccountId__count', 'users__AccountId__sum', 'users__DisplayName__count', 'users__Location__count', 'users__ProfileImageUrl__mean', 'users__ProfileImageUrl__count', 'users__ProfileImageUrl__sum', 'users__WebsiteUrl__count', 'users__AboutMe__count', 'users__CreationDate__min', 'users__CreationDate__max', 'posts__PostTypeId__mean', 'posts__PostTypeId__count', 'posts__PostTypeId__sum', 'posts__OwnerDisplayName__count', 'posts__Title__count', 'posts__Tags__count', 'posts__ContentLicense__count', 'p

,OwnerUserId,timestamp,contribution,users__AccountId__mean,users__AccountId__count,users__AccountId__sum,users__DisplayName__count,users__Location__count,users__ProfileImageUrl__mean,users__ProfileImageUrl__count,...,badges__TagBased__mean,badges__TagBased__count,badges__TagBased__sum,badges__Date__min,badges__Date__max,comments__ContentLicense__count,comments__UserDisplayName__count,comments__Text__count,comments__CreationDate__min,comments__CreationDate__max
0,352,2012-01-12,1,229230.0,1,229230.0,1,1,NaN,0,...,0.020833,48,1.0,2010-07-27 12:18:44.327,2012-01-08 18:31:28.823,353,0,353,2010-08-20 12:42:10.483,2012-01-09 18:26:30.950
1,7,2010-10-14,1,70002.0,1,70002.0,1,1,NaN,0,...,0.000000,25,0.0,2010-07-19 19:39:07.330,2010-10-13 16:59:17.470,72,0,72,2010-07-19 19:56:18.490,2010-10-13 09:05:34.927


In [16]:
# Build feature matrices for val/test too, and persist all three to disk.
import os
os.makedirs("features", exist_ok=True)

fm_val  = build_feature_matrix(val_table.df,  db, max_depth=2)
fm_test = build_feature_matrix(test_table.df, db, max_depth=2)

fm_train.to_pickle("features/fm_train.pkl")
fm_val.to_pickle("features/fm_val.pkl")
fm_test.to_pickle("features/fm_test.pkl")

print(f"\nsaved:")
print(f"  fm_train: {fm_train.shape}")
print(f"  fm_val:   {fm_val.shape}")
print(f"  fm_test:  {fm_test.shape}")

[depth-1] target ⟕ users ...
[depth-2] users ⋈ posts (on OwnerUserId) ... matched rows: 319,820
[depth-2] users ⋈ postHistory (on UserId) ... matched rows: 1,066,932
[depth-2] users ⋈ votes (on UserId) ... matched rows: 5,017
[depth-2] users ⋈ badges (on UserId) ... matched rows: 367,072
[depth-2] users ⋈ comments (on UserId) ... matched rows: 594,598
feature_matrix shape: (85838, 54)   (2.3s)
[depth-1] target ⟕ users ...
[depth-2] users ⋈ posts (on OwnerUserId) ... matched rows: 328,648
[depth-2] users ⋈ postHistory (on UserId) ... matched rows: 1,098,857
[depth-2] users ⋈ votes (on UserId) ... matched rows: 5,182
[depth-2] users ⋈ badges (on UserId) ... matched rows: 380,489
[depth-2] users ⋈ comments (on UserId) ... matched rows: 612,288
feature_matrix shape: (88137, 53)   (2.1s)

saved:
  fm_train: (1360850, 54)
  fm_val:   (85838, 54)
  fm_test:  (88137, 53)


## Q3 — How many features at each depth

| max_depth | # new features |
|---|---|
| 1 | **12** |
| 2 | **51** |
| 3 | **111** |

For each path ending in some leaf table T, the leaf contributes `(3 × num_numeric_cols) + (1 × num_text_cols) + (2 × num_timestamp_cols)` features.

Here's how each table breaks down (after skipping its PK and FK columns):
- `users` → 12 (AccountId 3, DisplayName 1, Location 1, ProfileImageUrl 3, WebsiteUrl 1, AboutMe 1, CreationDate 2)
- `posts` → 10, `postHistory` → 10, `votes` → 5, `badges` → 9, `comments` → 5, `postLinks` → 5

At depth 1 we have just `target ⟕ users` → 12 features.

At depth 2 we add 5 paths: `users ⋈ posts`, `users ⋈ postHistory`, `users ⋈ votes`, `users ⋈ badges`, `users ⋈ comments`. Adding contributions: 12 + 10 + 10 + 5 + 9 + 5 = **51**.

At depth 3 we add 8 more length-2 paths from users (e.g. `users ⋈ posts ⋈ postLinks` via each of the two postLinks FK columns, `users ⋈ comments ⋈ posts`, etc.). Their leaf contributions sum to 60, so total = 51 + 60 = **111**.

## Q4 — Shape of the feature matrix

`fm_train.shape = (1,360,850, 54)`.

The original target table was `(1,360,850, 3)` — just OwnerUserId, timestamp, and contribution. Our feature matrix has the same number of rows (because we used a LEFT JOIN to the target, so no rows are dropped), plus 51 new feature columns. So 3 + 51 = 54 columns total.

One thing worth noting: some of the intermediate joins were huge (the merge with postHistory created about 16.8M rows in memory), but the groupby step collapses everything back down to one row per `(user, timestamp)` target row, so the output stays at 1.36M rows.

## Q5 — Three example features

- **`posts__CreationDate__max`**: the timestamp of the user's most recent post (at or before the target timestamp). This captures how recently the user has posted. NaN if the user never posted.
- **`badges__Class__mean`**: average badge class for badges the user has earned. Badge class is 1/2/3 with 1 being the most prestigious, so this kind of captures the average quality of recognition the user has received rather than just the volume.
- **`comments__Text__count`**: the number of non-null `Text` fields in the user's comments, which is basically just the number of comments they have authored.

## Q6 — Missing values

Yes, lots of missing values. They come from a few different places:

1. **No matching rows after the join + temporal cutoff.** If a user has never commented before the timestamp, the aggregations over comments are NaN. (We explicitly set the `count` aggregations to 0 since "no rows" is meaningfully zero count, but mean/sum/min/max stay NaN.)
2. **NULL values in the source data.** `votes.UserId` is mostly NULL in this dataset because votes are anonymized. Several users columns (like `ProfileImageUrl`, `Location`) are also sparse.
3. **Aggregating over all-NaN inputs** just gives NaN back.

We think the missingness pattern itself is signal here - "this user has never voted before this timestamp" tells us something. Tree-based models (DT, XGBoost) handle NaN natively. For LogReg and the NN we needed to impute (we used median for the numeric features).

## Q7 — Train and compare four classifiers

We trained four models on the feature matrix from Q2 (51 features at max_depth=2):
- **Decision Tree** (sklearn)
- **XGBoost**
- **Logistic Regression** (sklearn)
- A small **MLP in PyTorch**

The target is the `contribution` column. We use the train/val/test split that came from `task.get_table(...)` since it's already split by timestamp.

**Preprocessing:**
- Drop the non-feature columns (`OwnerUserId`, `timestamp`, `contribution`)
- Convert datetime columns (`*__min`, `*__max`) to int64 nanoseconds since epoch, with NaN replaced by -1
- For LogReg and the NN: median impute then standardize with `StandardScaler` (we need this because these models are scale-sensitive)
- For Decision Tree and XGBoost: leave NaN as is (well, sklearn DT under 1.3 doesn't accept NaN so we `fillna(-1)` for that one specifically)

**Class imbalance:** train has about 5% positives, val about 3%. If we don't handle this the models would just predict 0 for everything. We used:
- DT, LR: `class_weight='balanced'`
- XGBoost: `scale_pos_weight = (#neg / #pos) ≈ 19`
- NN: `pos_weight` in `BCEWithLogitsLoss`

**Hyperparameter search:** a small grid per model (see the per-model cells below). For each model we picked the variant with the highest validation ROC-AUC, but we kept all variants logged in `all_runs` so the full grid is available.

In [17]:
# ---------- Preprocessing for Q7 ----------
import numpy as np
import pandas as pd

def prepare_xy(fm, drop_cols=("OwnerUserId", "timestamp", "contribution")):
    """Return (X DataFrame with numeric features only, y array or None)."""
    y = fm["contribution"].values.astype("int64") if "contribution" in fm.columns else None
    X = fm.drop(columns=[c for c in drop_cols if c in fm.columns]).copy()
    # convert datetime cols to int64 ns (NaN -> -1)
    for c in X.columns:
        if pd.api.types.is_datetime64_any_dtype(X[c]):
            X[c] = X[c].astype("int64")  # NaT becomes a large negative; we'll normalize below
            X[c] = X[c].mask(X[c] < 0, -1)
        elif X[c].dtype == "object":
            # shouldn't happen after our aggregations, but guard
            X[c] = pd.to_numeric(X[c], errors="coerce")
        else:
            X[c] = X[c].astype("float64")
    return X, y

X_train, y_train = prepare_xy(fm_train)
X_val,   y_val   = prepare_xy(fm_val)

print("X_train:", X_train.shape, " y_train pos-rate:", y_train.mean().round(4))
print("X_val:  ", X_val.shape,   " y_val pos-rate:  ", y_val.mean().round(4))
print("dtypes:", X_train.dtypes.value_counts().to_dict())
print("nan share per col (top 5):")
print(X_train.isna().mean().sort_values(ascending=False).head(5))

X_train: (1360850, 51)  y_train pos-rate: 0.05
X_val:   (85838, 51)  y_val pos-rate:   0.0281
dtypes: {dtype('float64'): 39, dtype('int64'): 12}
nan share per col (top 5):
users__ProfileImageUrl__mean    1.000000
users__ProfileImageUrl__sum     1.000000
votes__VoteTypeId__mean         0.969956
votes__VoteTypeId__sum          0.969956
badges__Class__sum              0.087756
dtype: float64


In [18]:
# ---------- Eval harness ----------
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
)

def eval_split(y_true, y_pred, y_score):
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "roc_auc":   roc_auc_score(y_true, y_score),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
    }

def fmt(metrics_tr, metrics_val):
    return {k: f"{metrics_tr[k]:.3f}/{metrics_val[k]:.3f}" for k in metrics_tr}

# Big dict to collect every variant we try, for the per-model variant tables in the report.
all_runs = []
def log_run(model_name, variant, m_tr, m_val):
    all_runs.append({"model": model_name, "variant": variant,
                     **{f"tr_{k}": v for k, v in m_tr.items()},
                     **{f"val_{k}": v for k, v in m_val.items()}})

In [19]:
# ---------- Decision Tree ----------
from sklearn.tree import DecisionTreeClassifier

# DT in sklearn < 1.3 does NOT handle NaN. Fill with -1 sentinel.
X_train_dt = X_train.fillna(-1).values
X_val_dt   = X_val.fillna(-1).values

dt_grid = [
    {"max_depth": 5},
    {"max_depth": 10},
    {"max_depth": 20},
    {"max_depth": None, "min_samples_leaf": 50},
]
best_dt, best_dt_val_auc = None, -1
for params in dt_grid:
    clf = DecisionTreeClassifier(class_weight="balanced", random_state=0, **params)
    clf.fit(X_train_dt, y_train)
    s_tr  = clf.predict_proba(X_train_dt)[:, 1]
    s_val = clf.predict_proba(X_val_dt)[:, 1]
    p_tr  = (s_tr  >= 0.5).astype(int)
    p_val = (s_val >= 0.5).astype(int)
    m_tr  = eval_split(y_train, p_tr,  s_tr)
    m_val = eval_split(y_val,   p_val, s_val)
    log_run("DecisionTree", str(params), m_tr, m_val)
    print(f"DT {params}: val AUC={m_val['roc_auc']:.3f}, val F1={m_val['f1']:.3f}")
    if m_val["roc_auc"] > best_dt_val_auc:
        best_dt, best_dt_val_auc = clf, m_val["roc_auc"]
        best_dt_params, best_dt_m_tr, best_dt_m_val = params, m_tr, m_val

print("\nBEST DT:", best_dt_params, "val AUC =", round(best_dt_val_auc, 3))

DT {'max_depth': 5}: val AUC=0.846, val F1=0.143
DT {'max_depth': 10}: val AUC=0.841, val F1=0.147
DT {'max_depth': 20}: val AUC=0.618, val F1=0.182
DT {'max_depth': None, 'min_samples_leaf': 50}: val AUC=0.846, val F1=0.155

BEST DT: {'max_depth': 5} val AUC = 0.846


In [20]:
# ---------- XGBoost ----------
from xgboost import XGBClassifier

pos_ratio = float((y_train == 0).sum() / (y_train == 1).sum())  # ~19 for ~5% positive

xgb_grid = [
    {"max_depth": 4,  "n_estimators": 200, "learning_rate": 0.10},
    {"max_depth": 6,  "n_estimators": 200, "learning_rate": 0.10},
    {"max_depth": 8,  "n_estimators": 400, "learning_rate": 0.05},
]
best_xgb, best_xgb_val_auc = None, -1
for params in xgb_grid:
    clf = XGBClassifier(
        scale_pos_weight=pos_ratio,
        tree_method="hist", n_jobs=-1, random_state=0,
        eval_metric="logloss", **params,
    )
    clf.fit(X_train.values, y_train)
    s_tr  = clf.predict_proba(X_train.values)[:, 1]
    s_val = clf.predict_proba(X_val.values)[:, 1]
    p_tr  = (s_tr  >= 0.5).astype(int)
    p_val = (s_val >= 0.5).astype(int)
    m_tr  = eval_split(y_train, p_tr,  s_tr)
    m_val = eval_split(y_val,   p_val, s_val)
    log_run("XGBoost", str(params), m_tr, m_val)
    print(f"XGB {params}: val AUC={m_val['roc_auc']:.3f}, val F1={m_val['f1']:.3f}")
    if m_val["roc_auc"] > best_xgb_val_auc:
        best_xgb, best_xgb_val_auc = clf, m_val["roc_auc"]
        best_xgb_params, best_xgb_m_tr, best_xgb_m_val = params, m_tr, m_val

print("\nBEST XGB:", best_xgb_params, "val AUC =", round(best_xgb_val_auc, 3))

XGB {'max_depth': 4, 'n_estimators': 200, 'learning_rate': 0.1}: val AUC=0.882, val F1=0.151
XGB {'max_depth': 6, 'n_estimators': 200, 'learning_rate': 0.1}: val AUC=0.879, val F1=0.156
XGB {'max_depth': 8, 'n_estimators': 400, 'learning_rate': 0.05}: val AUC=0.867, val F1=0.169

BEST XGB: {'max_depth': 4, 'n_estimators': 200, 'learning_rate': 0.1} val AUC = 0.882


In [21]:
# ---------- Logistic Regression ----------
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

imp = SimpleImputer(strategy="median").fit(X_train.values)
X_tr_imp = imp.transform(X_train.values)
X_va_imp = imp.transform(X_val.values)
scaler = StandardScaler().fit(X_tr_imp)
X_tr_sc = scaler.transform(X_tr_imp)
X_va_sc = scaler.transform(X_va_imp)

lr_grid = [
    {"C": 0.1},
    {"C": 1.0},
    {"C": 10.0},
]
best_lr, best_lr_val_auc = None, -1
for params in lr_grid:
    clf = LogisticRegression(
        class_weight="balanced", max_iter=2000, solver="lbfgs",
        n_jobs=-1, random_state=0, **params,
    )
    clf.fit(X_tr_sc, y_train)
    s_tr  = clf.predict_proba(X_tr_sc)[:, 1]
    s_val = clf.predict_proba(X_va_sc)[:, 1]
    p_tr  = (s_tr  >= 0.5).astype(int)
    p_val = (s_val >= 0.5).astype(int)
    m_tr  = eval_split(y_train, p_tr,  s_tr)
    m_val = eval_split(y_val,   p_val, s_val)
    log_run("LogReg", str(params), m_tr, m_val)
    print(f"LR {params}: val AUC={m_val['roc_auc']:.3f}, val F1={m_val['f1']:.3f}")
    if m_val["roc_auc"] > best_lr_val_auc:
        best_lr, best_lr_val_auc = clf, m_val["roc_auc"]
        best_lr_params, best_lr_m_tr, best_lr_m_val = params, m_tr, m_val

print("\nBEST LR:", best_lr_params, "val AUC =", round(best_lr_val_auc, 3))

/home/abed/miniconda3/envs/structml1/lib/python3.11/site-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [5 7]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/home/abed/miniconda3/envs/structml1/lib/python3.11/site-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [5 7]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


LR {'C': 0.1}: val AUC=0.824, val F1=0.160
LR {'C': 1.0}: val AUC=0.824, val F1=0.160
LR {'C': 10.0}: val AUC=0.823, val F1=0.160

BEST LR: {'C': 0.1} val AUC = 0.824


In [22]:
# ---------- Custom NN (PyTorch MLP) ----------
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("using device:", device)

# reuse imputed+scaled features from the LR cell (X_tr_sc, X_va_sc)
class MLP(nn.Module):
    def __init__(self, in_dim, hidden=(128, 64), dropout=0.2):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_mlp(hidden, dropout, lr, epochs=4, batch_size=4096):
    torch.manual_seed(0)
    Xt = torch.tensor(X_tr_sc, dtype=torch.float32)
    yt = torch.tensor(y_train, dtype=torch.float32)
    Xv = torch.tensor(X_va_sc, dtype=torch.float32).to(device)
    yv = torch.tensor(y_val,   dtype=torch.float32).to(device)
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True)

    model = MLP(Xt.shape[1], hidden=hidden, dropout=dropout).to(device)
    pos_w = torch.tensor([(y_train == 0).sum() / max((y_train == 1).sum(), 1)], dtype=torch.float32, device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            s_val = torch.sigmoid(model(Xv)).cpu().numpy()
        print(f"  epoch {ep+1}: val AUC={roc_auc_score(y_val, s_val):.3f}")
    return model

def score_mlp(model, X_np):
    model.eval()
    with torch.no_grad():
        scores = []
        for i in range(0, len(X_np), 65536):
            xb = torch.tensor(X_np[i:i+65536], dtype=torch.float32, device=device)
            scores.append(torch.sigmoid(model(xb)).cpu().numpy())
    return np.concatenate(scores)

nn_grid = [
    {"hidden": (128, 64), "dropout": 0.2, "lr": 1e-3, "epochs": 4},
    {"hidden": (256, 128, 64), "dropout": 0.3, "lr": 1e-3, "epochs": 4},
]
best_nn, best_nn_val_auc = None, -1
for params in nn_grid:
    print("NN", params)
    model = train_mlp(**params)
    s_tr  = score_mlp(model, X_tr_sc)
    s_val = score_mlp(model, X_va_sc)
    p_tr  = (s_tr  >= 0.5).astype(int)
    p_val = (s_val >= 0.5).astype(int)
    m_tr  = eval_split(y_train, p_tr,  s_tr)
    m_val = eval_split(y_val,   p_val, s_val)
    log_run("MLP", str(params), m_tr, m_val)
    print(f"  -> val AUC={m_val['roc_auc']:.3f}, val F1={m_val['f1']:.3f}\n")
    if m_val["roc_auc"] > best_nn_val_auc:
        best_nn, best_nn_val_auc = model, m_val["roc_auc"]
        best_nn_params, best_nn_m_tr, best_nn_m_val = params, m_tr, m_val

print("BEST NN:", best_nn_params, "val AUC =", round(best_nn_val_auc, 3))

using device: cuda
NN {'hidden': (128, 64), 'dropout': 0.2, 'lr': 0.001, 'epochs': 4}
  epoch 1: val AUC=0.863
  epoch 2: val AUC=0.867
  epoch 3: val AUC=0.867
  epoch 4: val AUC=0.869
  -> val AUC=0.869, val F1=0.150

NN {'hidden': (256, 128, 64), 'dropout': 0.3, 'lr': 0.001, 'epochs': 4}
  epoch 1: val AUC=0.867
  epoch 2: val AUC=0.871
  epoch 3: val AUC=0.869
  epoch 4: val AUC=0.871
  -> val AUC=0.871, val F1=0.148

BEST NN: {'hidden': (256, 128, 64), 'dropout': 0.3, 'lr': 0.001, 'epochs': 4} val AUC = 0.871


In [23]:
# ---------- Final summary tables ----------
summary_rows = [
    {"model": "Decision Tree",       **fmt(best_dt_m_tr,  best_dt_m_val)},
    {"model": "XGBoost",             **fmt(best_xgb_m_tr, best_xgb_m_val)},
    {"model": "Logistic Regression", **fmt(best_lr_m_tr,  best_lr_m_val)},
    {"model": "Custom NN (MLP)",     **fmt(best_nn_m_tr,  best_nn_m_val)},
]
summary = pd.DataFrame(summary_rows).set_index("model")
print("=== Q7 best-variant summary (train/val) ===")
print(summary)

print("\n=== Q7 full hyperparam grid (every variant tried) ===")
runs_df = pd.DataFrame(all_runs)
# round numeric cols
for c in runs_df.columns:
    if pd.api.types.is_numeric_dtype(runs_df[c]):
        runs_df[c] = runs_df[c].round(3)
print(runs_df.to_string(index=False))

# persist for the report
summary.to_csv("features/q7_summary.csv")
runs_df.to_csv("features/q7_all_runs.csv", index=False)

=== Q7 best-variant summary (train/val) ===
                        accuracy      roc_auc    precision       recall  \
model                                                                     
Decision Tree        0.772/0.733  0.806/0.846  0.141/0.079  0.698/0.793   
XGBoost              0.786/0.727  0.851/0.882  0.156/0.083  0.741/0.863   
Logistic Regression  0.792/0.784  0.810/0.824  0.149/0.090  0.671/0.730   
Custom NN (MLP)      0.773/0.731  0.840/0.871  0.147/0.081  0.733/0.836   

                              f1  
model                             
Decision Tree        0.234/0.143  
XGBoost              0.257/0.151  
Logistic Regression  0.244/0.160  
Custom NN (MLP)      0.244/0.148  

=== Q7 full hyperparam grid (every variant tried) ===
       model                                                              variant  tr_accuracy  tr_roc_auc  tr_precision  tr_recall  tr_f1  val_accuracy  val_roc_auc  val_precision  val_recall  val_f1
DecisionTree                           

## Q8 — What each metric means and which ones matter most

- **Accuracy**: fraction of predictions that are correct overall (`(TP+TN)/N`).
- **ROC AUC**: probability that a random positive example gets a higher score than a random negative example. It's threshold-independent.
- **Precision**: out of the users we predicted will engage, how many actually did (`TP / (TP+FP)`).
- **Recall**: out of the users who actually engaged, how many we caught (`TP / (TP+FN)`).
- **F1**: harmonic mean of precision and recall (so it balances them, at the chosen threshold).

**Tradeoffs between them:**

- Accuracy is misleading when classes are imbalanced. With 5% positives, you could just predict "0" for everyone and get 95% accuracy with 0 useful information.
- Precision and recall trade off against each other. Lowering the decision threshold catches more positives (↑ recall) but also calls more negatives positive by mistake (↓ precision). F1 is one point on this tradeoff curve.
- ROC AUC doesn't depend on a chosen threshold, which makes it the most reliable metric for comparing models, especially when classes are imbalanced.

**For our task** we think ROC AUC matters the most because:
1. It's robust to class imbalance.
2. It's the same metric the relbench paper uses, so we can compare results.
3. It doesn't force us to pick a deployment threshold up front.

Accuracy is the least useful here because of the imbalance. F1, precision and recall give a more concrete picture at a specific threshold (we used 0.5), but we mostly trust AUC for the head-to-head model comparison.

## Q9 — Discussion of results

The val ROC AUC ranking we got: **XGBoost 0.882 > NN 0.871 > Decision Tree 0.846 > LogReg 0.824**.

All four beat the 0.65 baseline from the relbench paper, so adding these 51 join-aggregate features really helped vs. using just the users table.

**Why XGBoost won:** boosted trees model non-linear interactions between features (like "many posts AND recent comment activity"), handle NaN natively (no need to impute), and they resist overfitting through shrinkage and depth caps. Our features (counts, timestamps, all numeric) are exactly what gradient-boosted trees do well with.

**The NN was a close second:** with standardized features and dropout, a 2-3 layer MLP can capture similar non-linearities. The reason it doesn't quite beat XGBoost is probably that the NN is more sensitive to imputation and scaling choices and doesn't get tree-style feature selection automatically.

**Decision Tree did worse than XGBoost:** with a single tree we have much higher variance. We saw this very clearly with `max_depth=20`, which got 0.95 train AUC but only 0.62 val AUC - textbook overfitting. The best variant was actually `max_depth=5` which underfits a bit but generalizes better.

**Logistic Regression did worst:** it's a linear model, so it can only learn linear relationships. A good example of where this hurts: "days since last comment" probably has a non-monotone effect (very recent = active, very old = inactive, but "never commented" should be its own thing). LR can only fit a straight line through this.

One unusual thing we noticed: validation AUC was *higher* than training AUC for DT, XGBoost, and the NN. We think this is because the val set comes later in time and has a lower positive rate (3% vs 5%), so the recently-active users who appear in val are easier to score correctly compared to the wider mix of users across all training time periods.

**How the schema affects things:** since the target table only directly connects to `users`, all other tables contribute through user history aggregates. The tables with the most rows per user (postHistory, badges, comments) end up giving the most useful features. `postLinks` isn't reachable from the target within 2 hops, so it contributes nothing at depth 2 - we'd need depth 3 to surface that signal.

## Q10 — How feature importance is computed in each model

- **Decision Tree (sklearn):** for each feature, sklearn sums up the Gini impurity reduction at every split that uses that feature, weighted by the number of training samples reaching that node, then normalizes so the importances sum to 1. So a feature is "important" if it gets used at splits that cleanly separate lots of samples.
- **XGBoost:** the default `feature_importances_` uses "gain". For each feature, sum the improvement in the loss function from every split that used that feature, across all the boosted trees. Then normalize. So it rewards features that consistently reduce the loss, not just features that get split on a lot. XGBoost also has alternatives like "weight" (just count of splits using the feature) and "cover" (avg samples per split) available through `get_booster().get_score(importance_type=...)`.
- **Logistic Regression:** sklearn's LR doesn't actually have a `feature_importances_` attribute. It has `coef_` instead. Because we standardized our features before fitting, the absolute value of each coefficient `|coef_[j]|` is directly comparable across features. A larger `|coef_[j]|` means a larger effect on the log-odds for a one-sigma change in feature j.

## Q11 — Top 10 features per model

In [24]:
# ---------- Q10/Q11: feature importances ----------
feature_names = list(X_train.columns)

def top_k(importances, k=10):
    idx = np.argsort(importances)[::-1][:k]
    return pd.DataFrame({"feature": [feature_names[i] for i in idx],
                         "importance": importances[idx]})

dt_top  = top_k(best_dt.feature_importances_)
xgb_top = top_k(best_xgb.feature_importances_)
lr_top  = top_k(np.abs(best_lr.coef_[0]))

print("=== DT top-10 ===");  print(dt_top.to_string(index=False))
print("\n=== XGB top-10 ==="); print(xgb_top.to_string(index=False))
print("\n=== LR top-10 (|coef| on standardized features) ==="); print(lr_top.to_string(index=False))

# Save side-by-side for the report
top10 = pd.concat({
    "Decision Tree": dt_top.reset_index(drop=True),
    "XGBoost":       xgb_top.reset_index(drop=True),
    "LogReg":        lr_top.reset_index(drop=True),
}, axis=1)
top10.to_csv("features/q11_top10.csv")

=== DT top-10 ===
                     feature  importance
          posts__Body__count    0.760768
    posts__PostTypeId__count    0.073188
       users__AboutMe__count    0.045879
 comments__CreationDate__max    0.029398
    posts__CreationDate__max    0.027742
         badges__Class__mean    0.015895
           badges__Date__max    0.015768
       comments__Text__count    0.009870
posts__ContentLicense__count    0.009711
       users__AccountId__sum    0.004589

=== XGB top-10 ===
                              feature  importance
             posts__PostTypeId__count    0.436050
               posts__PostTypeId__sum    0.169172
      comments__ContentLicense__count    0.051439
                users__AboutMe__count    0.044727
postHistory__PostHistoryTypeId__count    0.034612
               users__Location__count    0.031036
          comments__CreationDate__max    0.029182
       postHistory__CreationDate__max    0.026500
                  badges__Class__mean    0.021272
  postHisto

### Q11 — Discussion of top-10 features

Here are the top 10 features for each model, side by side (numbers from the cell above):

| Rank | Decision Tree | XGBoost | Logistic Regression |
|---|---|---|---|
| 1 | `posts__Body__count` (0.76) | `posts__PostTypeId__count` (0.44) | `badges__Class__mean` (9.76) |
| 2 | `posts__PostTypeId__count` (0.07) | `posts__PostTypeId__sum` (0.17) | `badges__TagBased__count` (6.19) |
| 3 | `users__AboutMe__count` (0.05) | `comments__ContentLicense__count` (0.05) | `badges__TagBased__sum` (5.48) |
| 4 | `comments__CreationDate__max` (0.029) | `users__AboutMe__count` (0.04) | `badges__Class__count` (3.22) |
| 5 | `posts__CreationDate__max` (0.028) | `postHistory__PostHistoryTypeId__count` (0.035) | `badges__Name__count` (3.22) |
| 6 | `badges__Class__mean` (0.016) | `users__Location__count` (0.031) | `votes__CreationDate__max` (3.22) |
| 7 | `badges__Date__max` (0.016) | `comments__CreationDate__max` (0.029) | `comments__Text__count` (3.04) |
| 8 | `comments__Text__count` (0.010) | `postHistory__CreationDate__max` (0.026) | `comments__UserDisplayName__count` (2.69) |
| 9 | `posts__ContentLicense__count` (0.010) | `badges__Class__mean` (0.021) | `postHistory__Comment__count` (1.96) |
| 10 | `users__AccountId__sum` (0.005) | `postHistory__PostHistoryTypeId__sum` (0.018) | `postHistory__Text__count` (1.92) |

**What's common across the three models.** All three rely heavily on user activity volume - counts of posts, comments, postHistory, or badges. `badges__Class__mean` actually shows up in the top 10 of all three models, and `comments__CreationDate__max` (recency of last comment) is in the top 10 of both Decision Tree and XGBoost. So our auto-generated features are surfacing the same kinds of signals we manually picked in Q1 (how much, how recently).

**Where they differ and why.**

The Decision Tree puts about 76% of its importance on a single feature: `posts__Body__count`, which is basically "how many posts the user has authored". This makes sense for a single tree - once it splits on the strongest feature at the root node, that split alone accounts for most of the impurity reduction in the whole tree. Other features that are very correlated with post count (like `posts__PostTypeId__count` or `posts__ContentLicense__count`) get very little credit because the tree already split on one of them.

XGBoost spreads importance much more evenly. The top feature is only at 44%, and the top 10 mixes post counts (44% + 17%), comment counts, postHistory counts, badges, and recency timestamps. The reason is that boosting fits trees sequentially on residuals, so once the strongest signal is partly explained, the next tree gets to use secondary features. This is probably also why XGBoost gets the best val AUC overall: it's using more of the signal in the feature matrix.

Logistic Regression's top 10 is dominated by badges - five out of the top six features are badge-related. We were a bit surprised by this. We think the reason is that badge columns have tight numerical ranges (Class is 1/2/3, TagBased is 0/1), so after standardization a one-sigma change is a big fraction of the feature's actual range, which makes the coefficient large in absolute value. Tree models don't care about the distribution of feature values, they only care about whether a split separates classes cleanly, which favors high-cardinality count features.

Also, LR mostly ignores recency timestamps - only `votes__CreationDate__max` makes its top 10. We think this is because recency vs. the label is non-monotonic in calendar time: very old timestamps (e.g. 2010) and very new timestamps (e.g. 2020) can both predict inactivity but for different reasons (account abandoned vs. brand new account). LR is linear so it can't bend the score back the way trees can.

**The takeaway:** the different importance metrics measure different things, and the same underlying signal (user activity) gets attributed to different features depending on the model. Trees pick one feature from a group of correlated ones and ignore the rest; LR distributes weight across them but weights them by their standardized variance. So getting divergent top-10 lists is expected here, and looking across the three is more informative than relying on any one of them.